In [1]:
%reset -f

In [2]:
%%capture
%pip install pandas_gbq

In [3]:
#installing packages
import pandas as pd
import numpy as np
from google.cloud import bigquery
import time
from datetime import timedelta
import json
import re
import itertools

import pandas_gbq
import matplotlib.pyplot as plt

In [4]:
#display settings
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)

In [5]:
#pull BQ data
sql_query = '''
SELECT * FROM `pcln-pl-busdatasci-prod.commercial_strategy.hotel_pcln_agoda_data_template_2023_2025_v19`;

'''

In [46]:
project_id = 'pcln-pl-busdatasci-prod'

# Initialize a BigQuery client
bq_client = bigquery.Client(project=project_id)

# Execute the query
job = bq_client.query(sql_query)
df = job.to_dataframe()

In [47]:
#CLEAN cig savings column
df["avg_cug_savings_pct"] = (
    df["avg_cug_savings_pct"]
      .replace("0E-9", 0)
      .astype(str)
)

df["avg_cug_savings_pct"] = pd.to_numeric(df["avg_cug_savings_pct"], errors="coerce")

In [48]:
# -------------------
# CONFIG
# -------------------
YEAR_MIN, YEAR_MAX = 2023, 2025
#DATASOURCES = ["PCLN"]
DATASOURCES = ["AGODA", "PCLN"]

BASE_KEYS = ["datasource", "submit_year"]

DIM_COLS = [
    "ap",
    "stay_segment",
    "location_segment",
    "purpose_segment",
    "country_segment",
    "guest_segment",
    "chain_scale_segment",
]

# Special % metrics
COL_APP_BOOK = "app_book"          # % units where == "APP"
COL_PROMO    = "promo_used_flag"   # % units where == "Y"
COL_PRODCLS  = "product_class"     # % units where == "SOPQ"

METRIC_UNITS = "gr_units"
METRIC_FEE   = "gr_fcst_contr_fee"
METRIC_BOOK  = "gr_book_amt"
METRIC_ORDER  = "gr_orders"
METRIC_PCT   = "avg_cug_savings_pct"

In [49]:
# Strip whitespace on object cols
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Normalize datasource and year
df["datasource"] = df["datasource"].str.upper()
df["submit_year"] = pd.to_numeric(df["submit_year"], errors="coerce")

# Coerce numeric metrics
for c in [METRIC_UNITS, METRIC_FEE, METRIC_BOOK, METRIC_ORDER, METRIC_PCT]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows missing key fields and remove packages
df = df.dropna(subset=["datasource", "submit_year"])
df = df[df["product_class"].astype(str).str.upper().str.strip() != "PKG"].copy()

In [50]:
# -------------------
# FILTER SCOPE
# -------------------
dff = df.loc[
    df["datasource"].isin(DATASOURCES) &
    df["submit_year"].between(YEAR_MIN, YEAR_MAX)
].copy()

print("Rows after datasource/year filter:", len(dff))

Rows after datasource/year filter: 332541


In [51]:
# -------------------
# PREP FOR WEIGHTED METRICS
# -------------------
# Weighted avg pct numerator: only rows where pct is present and units > 0
units_pos = dff[METRIC_UNITS].fillna(0).gt(0)
valid_pct = units_pos & dff[METRIC_PCT].notna()
dff["pct_x_units"] = np.where(valid_pct, dff[METRIC_PCT] * dff[METRIC_UNITS], np.nan)

# % units where app_book == "APP"
dff["app_units"] = np.where(
    units_pos & dff[COL_APP_BOOK].notna() & (dff[COL_APP_BOOK].astype(str).str.upper() == "APP"),
    dff[METRIC_UNITS],
    0.0
)

# % units where promo_used_flag == "Y"
dff["promo_units"] = np.where(
    units_pos & dff[COL_PROMO].notna() & (dff[COL_PROMO].astype(str).str.upper() == "Y"),
    dff[METRIC_UNITS],
    0.0
)

# % units where product_class == "SOPQ"
dff["sopq_units"] = np.where(
    units_pos & dff[COL_PRODCLS].notna() & (dff[COL_PRODCLS].astype(str).str.upper() == "SOPQ"),
    dff[METRIC_UNITS],
    0.0
)

In [12]:
# --------------------------------------------------
# FULL PERMUTATION AGGREGATION (ALL DIMENSIONS)
# --------------------------------------------------

BASE_KEYS = ["datasource", "submit_year"]
GROUP_COLS = BASE_KEYS + DIM_COLS

agg_df = (
    dff.groupby(GROUP_COLS, dropna=False)
       .agg(
           gr_units=("gr_units", "sum"),
           gr_fcst_contr_fee=("gr_fcst_contr_fee", "sum"),
           gr_book_amt=("gr_book_amt", "sum"),
           gr_orders=("gr_orders", "sum"),
           pct_units_sum=("pct_x_units", "sum"),
           app_units=("app_units", "sum"),
           promo_units=("promo_units", "sum"),
           sopq_units=("sopq_units", "sum"),
           avg_cug_savings_pct=("avg_cug_savings_pct", "mean"),  # keep consistent
       )
       .reset_index()
)

# -------------------------
# Recalculate % metrics correctly
# -------------------------
agg_df["pct_units_app"] = np.where(
    agg_df["gr_units"] > 0,
    agg_df["app_units"] / agg_df["gr_units"],
    np.nan
)

agg_df["pct_units_promo_y"] = np.where(
    agg_df["gr_units"] > 0,
    agg_df["promo_units"] / agg_df["gr_units"],
    np.nan
)

agg_df["pct_units_sopq"] = np.where(
    agg_df["gr_units"] > 0,
    agg_df["sopq_units"] / agg_df["gr_units"],
    np.nan
)

# Drop intermediate raw counts if desired
agg_df = agg_df.drop(
    columns=["pct_units_sum", "app_units", "promo_units", "sopq_units"]
)

In [13]:
## CAGR Calc

In [14]:
import warnings

In [15]:
METRICS = ["gr_units", "gr_fcst_contr_fee", "gr_book_amt", "gr_orders"]
YEARS = [2023, 2024, 2025]
START_YEAR, END_YEAR = 2023, 2025
N_YEARS = END_YEAR - START_YEAR  # 2 years between endpoints

# All segmentation dimensions
DIM_COLS = [
    "ap",
    "stay_segment",
    "location_segment",
    "purpose_segment",
    "country_segment",
    "guest_segment",
    "chain_scale_segment",
]

KEYS = ["datasource"] + DIM_COLS

dfg = agg_df.copy()

# -------------------------------------------------
# Clean & Filter
# -------------------------------------------------
dfg["submit_year"] = pd.to_numeric(dfg["submit_year"], errors="coerce").astype("Int64")
dfg = dfg[dfg["submit_year"].isin(YEARS)].copy()

for m in METRICS:
    dfg[m] = pd.to_numeric(dfg[m], errors="coerce")

# -------------------------------------------------
# 1) Simple CAGR (2023 → 2025)
# -------------------------------------------------
wide = dfg.pivot_table(
    index=KEYS,
    columns="submit_year",
    values=METRICS,
    aggfunc="sum"
)

wide.columns = [f"{metric}_{year}" for metric, year in wide.columns]
wide = wide.reset_index()

for m in METRICS:
    start = wide.get(f"{m}_{START_YEAR}")
    end   = wide.get(f"{m}_{END_YEAR}")

    wide[f"{m}_cagr_simple_{START_YEAR}_{END_YEAR}"] = np.where(
        (start > 0) & (end > 0),
        (end / start) ** (1 / N_YEARS) - 1,
        np.nan
    )

# -------------------------------------------------
# 2) Regression CAGR (log-linear)
# -------------------------------------------------
def reg_cagr_loglinear(g: pd.DataFrame, metric: str) -> float:
    x = g["submit_year"].astype(float).to_numpy()
    y = pd.to_numeric(g[metric], errors="coerce").to_numpy()

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    x = x[mask]
    y = y[mask]

    if x.size < 2:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", np.RankWarning)
        slope = np.polyfit(x, np.log(y), 1)[0]

    return float(np.exp(slope) - 1)

reg_rows = []
for keys, g in dfg.groupby(KEYS, dropna=False):
    row = dict(zip(KEYS, keys))
    for m in METRICS:
        row[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"] = reg_cagr_loglinear(g, m)
    reg_rows.append(row)

reg_df = pd.DataFrame(reg_rows)

# -------------------------------------------------
# Combine Outputs
# -------------------------------------------------
out = wide.merge(reg_df, on=KEYS, how="left")

# Optional: total growth implied by regression
for m in METRICS:
    ann = out[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"]
    out[f"{m}_cagr_reg_total_{START_YEAR}_{END_YEAR}"] = np.where(
        ann.notna(),
        (1 + ann) ** N_YEARS - 1,
        np.nan
    )

In [16]:
##Calc baseline

CAGR_FEE_COL   = f"gr_fcst_contr_fee_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_FEE_SIMPLE_COL   = f"gr_fcst_contr_fee_cagr_simple_{START_YEAR}_{END_YEAR}"

CAGR_UNITS_COL = f"gr_units_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_UNITS_SIMPLE_COL = f"gr_units_cagr_simple_{START_YEAR}_{END_YEAR}"

CAGR_TTV_COL   = f"gr_book_amt_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_TTV_SIMPLE_COL   = f"gr_book_amt_cagr_simple_{START_YEAR}_{END_YEAR}"

CAGR_ORDERS_COL   = f"gr_orders_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_ORDERS_SIMPLE_COL   = f"gr_orders_cagr_simple_{START_YEAR}_{END_YEAR}"

# =========================================================
# 0) Build PCLN baseline from full dataset (before filtering)
# =========================================================

base_2025 = agg_df.copy()
base_2025["submit_year"] = pd.to_numeric(base_2025["submit_year"], errors="coerce")

# PCLN + 2025 + ALL TOTALS across dimensions
base_2025 = base_2025[
    (base_2025["datasource"].str.upper() == "PCLN") &
    (base_2025["submit_year"] == END_YEAR)
].copy()

for c in DIM_COLS:
    base_2025 = base_2025[base_2025[c].astype(str).str.strip().str.upper() == "TOTAL"]

# CAGR baseline from regression output
base_cagr = out.copy()
base_cagr = base_cagr[
    (base_cagr["datasource"].str.upper() == "PCLN")
].copy()

for c in DIM_COLS:
    base_cagr = base_cagr[base_cagr[c].astype(str).str.strip().str.upper() == "TOTAL"]

baseline = {
    CAGR_UNITS_COL: base_cagr[CAGR_UNITS_COL].mean(),
    CAGR_TTV_COL:   base_cagr[CAGR_TTV_COL].mean(),
    "pct_units_app":       base_2025["pct_x_units"].mean() if "pct_x_units" in base_2025 else base_2025["pct_units_app"].mean(),
    "pct_units_promo_y":   base_2025["promo_units"].mean() if "promo_units" in base_2025 else base_2025["pct_units_promo_y"].mean(),
    "pct_units_sopq":      base_2025["sopq_units"].mean() if "sopq_units" in base_2025 else base_2025["pct_units_sopq"].mean(),
    "avg_cug_savings_pct": base_2025["avg_cug_savings_pct"].mean(),
}

missing_baselines = [k for k, v in baseline.items() if pd.isna(v)]
if missing_baselines:
    raise ValueError(f"Missing baseline values for: {missing_baselines}")

zero_baselines = [k for k, v in baseline.items()
                  if isinstance(v, (int, float, np.floating)) and v == 0]
if zero_baselines:
    raise ValueError(f"Baseline value is 0 for: {zero_baselines}")

In [17]:
baseline

{'gr_units_cagr_reg_annual_2023_2025': -0.04130652691440262,
 'gr_book_amt_cagr_reg_annual_2023_2025': -0.04986418171972662,
 'pct_units_app': 0.4459997791243265,
 'pct_units_promo_y': 0.03082076708829828,
 'pct_units_sopq': 0.06668331823102983,
 'avg_cug_savings_pct': 23.518}

### Build out composite scores

In [24]:
# =========================================================
# 1) Filter to PCLN + 2025 and join CAGR
# =========================================================
df_2025 = agg_df.copy()
df_2025["submit_year"] = pd.to_numeric(df_2025["submit_year"], errors="coerce")

df_2025 = df_2025[
    (df_2025["datasource"].str.upper() == "PCLN") &
    (df_2025["submit_year"] == END_YEAR)
].copy()

keep_cols = KEYS + [
    "gr_units",
    "gr_fcst_contr_fee",
    "gr_orders",
    "gr_book_amt",
    "pct_units_app",
    "pct_units_promo_y",
    "pct_units_sopq",
    "avg_cug_savings_pct",
]

df_2025 = df_2025[keep_cols].copy()

# Filter low producers
df_2025 = df_2025[df_2025["gr_units"].fillna(0) >= 10].copy()

# Join CAGR metrics
sc = df_2025.merge(
    out[KEYS + [
        CAGR_UNITS_COL,
        CAGR_UNITS_SIMPLE_COL,
        CAGR_FEE_COL,
        CAGR_FEE_SIMPLE_COL,
        CAGR_TTV_COL,
        CAGR_TTV_SIMPLE_COL,
        CAGR_ORDERS_COL,
        CAGR_ORDERS_SIMPLE_COL
    ]],
    on=KEYS,
    how="left"
)

# Remove TOTAL rows
#for c in DIM_COLS:
    #sc = sc[sc[c].astype(str).str.strip().str.upper() != "TOTAL"]

sc = sc.dropna(subset=[
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL, 
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
    CAGR_ORDERS_COL,
    CAGR_ORDERS_SIMPLE_COL,
    "pct_units_app", "pct_units_promo_y",
    "pct_units_sopq", "avg_cug_savings_pct"
])

sc = sc[sc["gr_units"].fillna(0) > 0].copy()

# Segment label
#sc["segment_label"] = sc[DIM_COLS].astype(str).agg(" | ".join, axis=1)

# Calculate % of Total
TOTAL_FILTER = (
    (sc["datasource"].str.upper() == "PCLN") &
    (sc["ap"] == "TOTAL") &
    (sc["stay_segment"] == "TOTAL") &
    (sc["location_segment"] == "TOTAL") &
    (sc["purpose_segment"] == "TOTAL") &
    (sc["country_segment"] == "TOTAL") &
    (sc["guest_segment"] == "TOTAL") &
    (sc["chain_scale_segment"] == "TOTAL")
)

total_row = sc.loc[TOTAL_FILTER].iloc[0]

TOTAL_METRICS = [
    "gr_units",
    "gr_fcst_contr_fee",
    "gr_orders",
    "gr_book_amt",
]

totals = sc.loc[TOTAL_FILTER, TOTAL_METRICS].iloc[0]

for col in TOTAL_METRICS:
    sc[f"pct_total_{col}"] = sc[col] / totals[col]


# =========================================================
# 2) Normalization vs PCLN baseline
# =========================================================

def ratio_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    if avg == 0 or pd.isna(avg):
        return pd.Series(np.nan, index=s.index)
    idx = s / avg
    if clip_low is not None or clip_high is not None:
        idx = idx.clip(lower=clip_low, upper=clip_high)
    return idx

def robust_diff_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    x = pd.to_numeric(s, errors="coerce")
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        std = np.nanstd(x)
        scale = std if std not in [0, np.nan] else 1.0
    else:
        scale = mad

    z = (x - avg) / scale
    if clip_low is not None or clip_high is not None:
        z = z.clip(lower=clip_low, upper=clip_high)
    return z

RATIO_CLIP_LOW, RATIO_CLIP_HIGH = 0.25, 4.0
DIFF_CLIP_LOW, DIFF_CLIP_HIGH   = -4.0, 4.0

# -----------------------------------------
# Blend SIMPLE + REGRESSION CAGR (RAW)
# -----------------------------------------

sc["gr_units_cagr_blend"] = np.nanmean(
    sc[[CAGR_UNITS_COL, CAGR_UNITS_SIMPLE_COL]],
    axis=1
)

sc["gr_ttv_cagr_blend"] = np.nanmean(
    sc[[CAGR_TTV_COL, CAGR_TTV_SIMPLE_COL]],
    axis=1
)

sc["gr_orders_cagr_blend"] = np.nanmean(
    sc[[CAGR_ORDERS_COL, CAGR_ORDERS_SIMPLE_COL]],
    axis=1
)

sc["gr_fee_cagr_blend"] = np.nanmean(
    sc[[CAGR_FEE_COL, CAGR_FEE_SIMPLE_COL]],
    axis=1
)


# Growth normalization (robust vs baseline)
sc["units_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_units_cagr_blend"], baseline[CAGR_UNITS_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

sc["ttv_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_ttv_cagr_blend"], baseline[CAGR_TTV_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

# Mix normalization (ratio vs baseline)
sc["app_norm"] = ratio_vs_avg(
    sc["pct_units_app"], baseline["pct_units_app"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["promo_norm"] = ratio_vs_avg(
    sc["pct_units_promo_y"], baseline["pct_units_promo_y"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["sopq_norm"] = ratio_vs_avg(
    sc["pct_units_sopq"], baseline["pct_units_sopq"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["cug_savings_norm"] = ratio_vs_avg(
    sc["avg_cug_savings_pct"], baseline["avg_cug_savings_pct"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

cols_to_drop = [
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL, 
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
    CAGR_ORDERS_COL,
    CAGR_ORDERS_SIMPLE_COL
]

sc = sc.drop(columns=cols_to_drop)


# =========================================================
# 3) Composite Scores
# =========================================================
w_units, w_fee = 0.5, 0.5
w_app = w_promo = w_sopq = w_cug = 1/4
w_growth, w_mix = 0.6, 0.4

sc["growth_score"] = (
    w_units * sc["units_cagr_norm"] +
    w_fee   * sc["ttv_cagr_norm"]
)

sc["moat_score"] = (
    w_app   * sc["app_norm"] +
    w_promo * sc["promo_norm"] +
    w_sopq  * sc["sopq_norm"] +
    w_cug   * sc["cug_savings_norm"]
)

sc["composite_score"] = (
    w_growth * sc["growth_score"] +
    w_mix    * sc["moat_score"]
)

### Output results

In [26]:
# -----------------------
# 6) Optional: view the underlying table
# -----------------------
top_segs=sc.sort_values("composite_score", ascending=False)

In [27]:
top_segs.to_csv("all_dimension_segs.csv", index=False)

In [28]:
sc.sort_values("gr_units_cagr_blend", ascending=False).head(10)

,datasource,ap,stay_segment,location_segment,purpose_segment,country_segment,guest_segment,chain_scale_segment,gr_units,gr_fcst_contr_fee,gr_orders,gr_book_amt,pct_units_app,pct_units_promo_y,pct_units_sopq,avg_cug_savings_pct,pct_total_gr_units,pct_total_gr_fcst_contr_fee,pct_total_gr_orders,pct_total_gr_book_amt,gr_units_cagr_blend,gr_ttv_cagr_blend,gr_orders_cagr_blend,gr_fee_cagr_blend,units_cagr_norm,ttv_cagr_norm,app_norm,promo_norm,sopq_norm,cug_savings_norm,growth_score,moat_score,composite_score
12223,PCLN,short,Extended Long Stay,Airport,Mixed / Leisure Travel,Asia,Solo,Midscale,"1,439.0000","16,084.1200",83.0000,"128,621.4200",0.0000,0.0000,0.0000,10.0000,0.0000,0.0000,0.0000,0.0000,6.9098,9.8073,8.1104,10.1554,4.0000,4.0000,0.2500,0.2500,0.2500,0.4252,4.0000,0.2938,2.5175
15216,PCLN,short,Weekday Short Stay,Interstate,Mixed Extended / Work Stay,US,Adult Group,Luxury,156.0000,"1,503.3900",121.0000,"11,476.5900",0.5962,0.0000,0.0000,10.0000,0.0000,0.0000,0.0000,0.0000,6.2111,5.9136,5.3509,4.8427,4.0000,4.0000,1.3367,0.2500,0.2500,0.4252,4.0000,0.5655,2.6262
17350,PCLN,short,Weekend Short Stay,Suburban,Logistics Industrial Data Business,CA,Solo,Upscale,52.0000,729.7200,42.0000,"5,248.7600",0.1154,0.1346,0.1538,32.0820,0.0000,0.0000,0.0000,0.0000,6.2111,5.2337,5.4807,8.1426,4.0000,4.0000,0.2587,4.0000,2.3071,1.3641,4.0000,1.9825,3.1930
6362,PCLN,mid_term,Extended Long Stay,Airport,Mixed / Leisure Travel,Asia,Solo,Midscale,"1,756.0000","16,992.3400",77.0000,"139,964.0100",0.0205,0.0000,0.0000,14.2300,0.0001,0.0000,0.0000,0.0000,6.1866,7.0173,5.2048,6.7380,4.0000,4.0000,0.2500,0.2500,0.2500,0.6051,4.0000,0.3388,2.5355
11393,PCLN,mid_term,Weekend Short Stay,Resort,Mixed Extended / Work Stay,Other,Adult Group,Economy,50.0000,"2,182.7200",24.0000,"16,195.0600",0.2600,0.0000,0.0000,10.4350,0.0000,0.0000,0.0000,0.0000,6.0711,6.3278,3.8990,6.4811,4.0000,4.0000,0.5830,0.2500,0.2500,0.4437,4.0000,0.3817,2.5527
2223,PCLN,long,Longer Weekend Stay,Airport,Mixed / Leisure Travel,Asia,Adult Group,Economy,183.0000,648.4900,30.0000,"4,819.2400",0.0000,0.0000,0.0000,8.0000,0.0000,0.0000,0.0000,0.0000,5.7639,3.5428,4.4772,4.3192,4.0000,4.0000,0.2500,0.2500,0.2500,0.3402,4.0000,0.2725,2.5090
8334,PCLN,mid_term,Longer Weekend Stay,Airport,Mixed / Leisure Travel,Asia,Solo,Midscale,"1,429.0000","9,936.3400",250.0000,"78,783.3600",0.0126,0.0000,0.0000,17.2500,0.0000,0.0000,0.0000,0.0000,5.6825,4.8034,5.4550,5.0047,4.0000,4.0000,0.2500,0.2500,0.2500,0.7335,4.0000,0.3709,2.5483
14820,PCLN,short,Longer Weekend Stay,Urban,Mixed Extended / Work Stay,US,Adult Group,Economy,174.0000,"4,413.8600",31.0000,"33,158.7600",0.4598,0.0000,0.0000,12.7850,0.0000,0.0000,0.0000,0.0000,5.5955,6.7973,4.5678,5.7053,4.0000,4.0000,1.0309,0.2500,0.2500,0.5436,4.0000,0.5186,2.6075
11082,PCLN,mid_term,Weekend Short Stay,Interstate,Mixed / Leisure Travel,US,Single-Room Family,Luxury,84.0000,"2,571.9400",57.0000,"20,046.6600",0.4881,0.0595,0.2976,38.7267,0.0000,0.0000,0.0000,0.0000,5.4807,8.3575,4.3385,7.9317,4.0000,4.0000,1.0944,1.9313,4.0000,1.6467,4.0000,2.1681,3.2672
5211,PCLN,long,Weekend Short Stay,Interstate,Mixed Extended / Work Stay,US,Adult Group,Luxury,75.0000,972.6900,34.0000,"7,424.9700",0.6000,0.0000,0.0000,10.0000,0.0000,0.0000,0.0000,0.0000,5.1237,5.9486,4.8310,5.9496,4.0000,4.0000,1.3453,0.2500,0.2500,0.4252,4.0000,0.5676,2.6270


In [52]:
# upload csv
segment_df = pd.read_csv('segments_dimension_lookup.csv', encoding='latin1')

In [53]:
DIM_COLS = [
    "ap",
    "stay_segment",
    "location_segment",
    "purpose_segment",
    "country_segment",
    "guest_segment",
    "chain_scale_segment",
]

dff = dff.merge(
    segment_df[DIM_COLS + ["segment"]],
    on=DIM_COLS,
    how="left"
)



In [54]:
# identify TOTAL row
TOTAL_FILTER = (
    (dff["ap"] == "TOTAL") &
    (dff["stay_segment"] == "TOTAL") &
    (dff["location_segment"] == "TOTAL") &
    (dff["purpose_segment"] == "TOTAL") &
    (dff["country_segment"] == "TOTAL") &
    (dff["guest_segment"] == "TOTAL") &
    (dff["chain_scale_segment"] == "TOTAL")
)

# overwrite segment
dff.loc[TOTAL_FILTER, "segment"] = "TOTAL"


In [56]:
dff["segment"] = dff["segment"].fillna("Rest of Market")

In [58]:
# --------------------------------------------------
# SEGEMENT AGGREGATION
# --------------------------------------------------

BASE_KEYS = ["datasource", "submit_year"]
GROUP_COLS = BASE_KEYS + ["segment"]

seg_agg_df = (
    dff.groupby(GROUP_COLS, dropna=False)
       .agg(
           gr_units=("gr_units", "sum"),
           gr_fcst_contr_fee=("gr_fcst_contr_fee", "sum"),
           gr_book_amt=("gr_book_amt", "sum"),
           gr_orders=("gr_orders", "sum"),
           pct_units_sum=("pct_x_units", "sum"),
           app_units=("app_units", "sum"),
           promo_units=("promo_units", "sum"),
           sopq_units=("sopq_units", "sum"),
           avg_cug_savings_pct=("avg_cug_savings_pct", "mean"),  # keep consistent
       )
       .reset_index()
)

# -------------------------
# Recalculate % metrics correctly
# -------------------------
seg_agg_df["pct_units_app"] = np.where(
    seg_agg_df["gr_units"] > 0,
    seg_agg_df["app_units"] / seg_agg_df["gr_units"],
    np.nan
)

seg_agg_df["pct_units_promo_y"] = np.where(
    seg_agg_df["gr_units"] > 0,
    seg_agg_df["promo_units"] / seg_agg_df["gr_units"],
    np.nan
)

seg_agg_df["pct_units_sopq"] = np.where(
    seg_agg_df["gr_units"] > 0,
    seg_agg_df["sopq_units"] / seg_agg_df["gr_units"],
    np.nan
)

# Drop intermediate raw counts if desired
seg_agg_df = seg_agg_df.drop(
    columns=["pct_units_sum", "app_units", "promo_units", "sopq_units"]
)

In [59]:
KEYS = ["datasource", "segment"]

dfg = seg_agg_df.copy()

# -------------------------------------------------
# Clean & Filter
# -------------------------------------------------
dfg["submit_year"] = pd.to_numeric(dfg["submit_year"], errors="coerce").astype("Int64")
dfg = dfg[dfg["submit_year"].isin(YEARS)].copy()

for m in METRICS:
    dfg[m] = pd.to_numeric(dfg[m], errors="coerce")

# -------------------------------------------------
# 1) Simple CAGR (2023 → 2025)
# -------------------------------------------------
wide = dfg.pivot_table(
    index=KEYS,
    columns="submit_year",
    values=METRICS,
    aggfunc="sum"
)

wide.columns = [f"{metric}_{year}" for metric, year in wide.columns]
wide = wide.reset_index()

for m in METRICS:
    start = wide.get(f"{m}_{START_YEAR}")
    end   = wide.get(f"{m}_{END_YEAR}")

    wide[f"{m}_cagr_simple_{START_YEAR}_{END_YEAR}"] = np.where(
        (start > 0) & (end > 0),
        (end / start) ** (1 / N_YEARS) - 1,
        np.nan
    )

# -------------------------------------------------
# 2) Regression CAGR (log-linear)
# -------------------------------------------------
def reg_cagr_loglinear(g: pd.DataFrame, metric: str) -> float:
    x = g["submit_year"].astype(float).to_numpy()
    y = pd.to_numeric(g[metric], errors="coerce").to_numpy()

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    x = x[mask]
    y = y[mask]

    if x.size < 2:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", np.RankWarning)
        slope = np.polyfit(x, np.log(y), 1)[0]

    return float(np.exp(slope) - 1)

reg_rows = []
for keys, g in dfg.groupby(KEYS, dropna=False):
    row = dict(zip(KEYS, keys))
    for m in METRICS:
        row[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"] = reg_cagr_loglinear(g, m)
    reg_rows.append(row)

reg_df = pd.DataFrame(reg_rows)

# -------------------------------------------------
# Combine Outputs
# -------------------------------------------------
out = wide.merge(reg_df, on=KEYS, how="left")

# Optional: total growth implied by regression
for m in METRICS:
    ann = out[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"]
    out[f"{m}_cagr_reg_total_{START_YEAR}_{END_YEAR}"] = np.where(
        ann.notna(),
        (1 + ann) ** N_YEARS - 1,
        np.nan
    )

In [61]:
# =========================================================
# 1) Filter to PCLN + 2025 and join CAGR
# =========================================================
df_2025 = seg_agg_df.copy()
df_2025["submit_year"] = pd.to_numeric(df_2025["submit_year"], errors="coerce")

df_2025 = df_2025[
    (df_2025["datasource"].str.upper() == "PCLN") &
    (df_2025["submit_year"] == END_YEAR)
].copy()

keep_cols = KEYS + [
    "gr_units",
    "gr_fcst_contr_fee",
    "gr_orders",
    "gr_book_amt",
    "pct_units_app",
    "pct_units_promo_y",
    "pct_units_sopq",
    "avg_cug_savings_pct",
]

df_2025 = df_2025[keep_cols].copy()

# Join CAGR metrics
sc = df_2025.merge(
    out[KEYS + [
        CAGR_UNITS_COL,
        CAGR_UNITS_SIMPLE_COL,
        CAGR_FEE_COL,
        CAGR_FEE_SIMPLE_COL,
        CAGR_TTV_COL,
        CAGR_TTV_SIMPLE_COL,
        CAGR_ORDERS_COL,
        CAGR_ORDERS_SIMPLE_COL
    ]],
    on=KEYS,
    how="left"
)

# Remove TOTAL rows
#for c in DIM_COLS:
    #sc = sc[sc[c].astype(str).str.strip().str.upper() != "TOTAL"]

sc = sc.dropna(subset=[
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL, 
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
    CAGR_ORDERS_COL,
    CAGR_ORDERS_SIMPLE_COL,
    "pct_units_app", "pct_units_promo_y",
    "pct_units_sopq", "avg_cug_savings_pct"
])

sc = sc[sc["gr_units"].fillna(0) > 0].copy()

# Segment label
#sc["segment_label"] = sc[DIM_COLS].astype(str).agg(" | ".join, axis=1)

# Calculate % of Total
TOTAL_FILTER = (
    (sc["datasource"].str.upper() == "PCLN") &
    (sc["segment"] == "TOTAL")
)

total_row = sc.loc[TOTAL_FILTER].iloc[0]

TOTAL_METRICS = [
    "gr_units",
    "gr_fcst_contr_fee",
    "gr_orders",
    "gr_book_amt",
]

totals = sc.loc[TOTAL_FILTER, TOTAL_METRICS].iloc[0]

for col in TOTAL_METRICS:
    sc[f"pct_total_{col}"] = sc[col] / totals[col]


# =========================================================
# 2) Normalization vs PCLN baseline
# =========================================================

def ratio_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    if avg == 0 or pd.isna(avg):
        return pd.Series(np.nan, index=s.index)
    idx = s / avg
    if clip_low is not None or clip_high is not None:
        idx = idx.clip(lower=clip_low, upper=clip_high)
    return idx

def robust_diff_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    x = pd.to_numeric(s, errors="coerce")
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        std = np.nanstd(x)
        scale = std if std not in [0, np.nan] else 1.0
    else:
        scale = mad

    z = (x - avg) / scale
    if clip_low is not None or clip_high is not None:
        z = z.clip(lower=clip_low, upper=clip_high)
    return z

RATIO_CLIP_LOW, RATIO_CLIP_HIGH = 0.25, 4.0
DIFF_CLIP_LOW, DIFF_CLIP_HIGH   = -4.0, 4.0

# -----------------------------------------
# Blend SIMPLE + REGRESSION CAGR (RAW)
# -----------------------------------------

sc["gr_units_cagr_blend"] = np.nanmean(
    sc[[CAGR_UNITS_COL, CAGR_UNITS_SIMPLE_COL]],
    axis=1
)

sc["gr_ttv_cagr_blend"] = np.nanmean(
    sc[[CAGR_TTV_COL, CAGR_TTV_SIMPLE_COL]],
    axis=1
)

sc["gr_orders_cagr_blend"] = np.nanmean(
    sc[[CAGR_ORDERS_COL, CAGR_ORDERS_SIMPLE_COL]],
    axis=1
)

sc["gr_fee_cagr_blend"] = np.nanmean(
    sc[[CAGR_FEE_COL, CAGR_FEE_SIMPLE_COL]],
    axis=1
)


# Growth normalization (robust vs baseline)
sc["units_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_units_cagr_blend"], baseline[CAGR_UNITS_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

sc["ttv_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_ttv_cagr_blend"], baseline[CAGR_TTV_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

# Mix normalization (ratio vs baseline)
sc["app_norm"] = ratio_vs_avg(
    sc["pct_units_app"], baseline["pct_units_app"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["promo_norm"] = ratio_vs_avg(
    sc["pct_units_promo_y"], baseline["pct_units_promo_y"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["sopq_norm"] = ratio_vs_avg(
    sc["pct_units_sopq"], baseline["pct_units_sopq"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["cug_savings_norm"] = ratio_vs_avg(
    sc["avg_cug_savings_pct"], baseline["avg_cug_savings_pct"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

cols_to_drop = [
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL, 
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
    CAGR_ORDERS_COL,
    CAGR_ORDERS_SIMPLE_COL
]

sc = sc.drop(columns=cols_to_drop)


# =========================================================
# 3) Composite Scores
# =========================================================
w_units, w_fee = 0.5, 0.5
w_app = w_promo = w_sopq = w_cug = 1/4
w_growth, w_mix = 0.6, 0.4

sc["growth_score"] = (
    w_units * sc["units_cagr_norm"] +
    w_fee   * sc["ttv_cagr_norm"]
)

sc["moat_score"] = (
    w_app   * sc["app_norm"] +
    w_promo * sc["promo_norm"] +
    w_sopq  * sc["sopq_norm"] +
    w_cug   * sc["cug_savings_norm"]
)

sc["composite_score"] = (
    w_growth * sc["growth_score"] +
    w_mix    * sc["moat_score"]
)

In [ ]:
sc["segment"] = sc["segment"].replace(
    "US Suburban Short-Stay Leisure (Declining)",
    "US Suburban Short-Stay Leisure"
)

In [ ]:
sc["segment"] = sc["segment"].replace(
    "Europe Mixed Demand",
    "Europe Mixed Travel"
)

In [76]:
sc.to_csv("pcln_segments.csv", index=False)

### Plotly outputs

In [63]:
import plotly.express as px

In [77]:
# normalize strings
sc["segment"] = sc["segment"].astype(str).str.strip()

segments = [
    "Airport Transit Stays",
    "Asia Planned Travel",
    "Asia Short-Lead Leisure",
    "Europe Mixed Travel",
    "Extended Stay Work Travel",
    "Group Leisure Travel",
    "North America Budget Road Travel",
    "Premium Resort Leisure",
    "Premium Urban Travel",
    "Resort Value Leisure",
    "Rest of Market",
    "Suburban Mixed Travel",
    "US Family Leisure Travel",
    "US Suburban Short-Stay Leisure",
    "US Weekday Business Travel",
    "US Weekend Leisure Travel",
]

conditions = [sc["segment"] == s for s in segments]
labels = segments

sc["highlight_group"] = np.select(conditions, labels, default="Other")

In [78]:
sc2 = sc[sc["highlight_group"] != "Other"].copy()

In [80]:
hover_cols = [
    "highlight_group",
    "gr_units",
    "gr_book_amt",
    "gr_units_cagr_blend",
    "gr_ttv_cagr_blend",
    "growth_score",
    "moat_score",
]

fig = px.scatter(
    sc2,
    x="growth_score",
    y="moat_score",
    size="gr_units",
    size_max=55,
    color="highlight_group",
    hover_name="segment",
    hover_data=hover_cols,
    title="PCLN: Growth vs 2025 Moat score",
)

fig.update_layout(
    legend_title_text="Highlight group",
    xaxis_title="Growth score (Units and TTV CAGR Scaled difference from PCLN avg)",
    yaxis_title="2025 Moat score (app %, SOPQ %, coupon %, CUG discount % indexed to PCLN avg)",
)

fig.write_html("pcln_growth_vs_moat_segments.html", include_plotlyjs="cdn")